In [0]:
from pyspark.sql.functions import (
    col, broadcast, lit, concat, floor, rand, count, sum as _sum,
    round as _round, avg, spark_partition_id, current_timestamp
)
import time

dbutils.widgets.text("catalog", "databricks_dev")
CATALOG = dbutils.widgets.get("catalog")

spark.sql(f"USE CATALOG {CATALOG}")

In [0]:

df_orders    = spark.table(f"{CATALOG}.silver.silver_orders")
df_items     = spark.table(f"{CATALOG}.silver.silver_order_items")
df_customers = spark.table(f"{CATALOG}.silver.silver_customers")
df_products  = spark.table(f"{CATALOG}.silver.silver_products")
df_fact      = spark.table(f"{CATALOG}.gold.fact_sales")


In [0]:
def time_query(label, func):
    """Run a function and measure execution time."""
    start = time.time()
    result = func()
    if hasattr(result, 'count'):
        result.count()
    elapsed = time.time() - start
    print(f"  {label}: {elapsed:.2f} seconds")
    return elapsed, result

In [0]:

# Force sort-merge join (no broadcast) using hint
t1, _ = time_query("WITHOUT Broadcast (SortMerge)", lambda:
    df_items
    .hint("merge")
    .join(df_products, "product_id", "inner")
)

# Check the plan — look for "SortMergeJoin" (expensive shuffle)
print("\n  Physical Plan (no broadcast):")
(df_items
 .join(df_products, "product_id", "inner")
).explain(mode="simple")


In [0]:
t2, _ = time_query("WITH Broadcast", lambda:
    df_items
    .join(broadcast(df_products), "product_id", "inner")
    .join(broadcast(df_customers).join(df_orders, "customer_id"),
          df_items["order_id"] == df_orders["order_id"], "inner")
)

In [0]:
print("\n  Physical Plan (with broadcast):")
(df_items
 .join(broadcast(df_products), "product_id", "inner")
).explain(mode="simple")

In [0]:
print(f"\n  Speedup: {t1/t2:.2f}x faster with broadcast")

In [0]:
print("Auto broadcast threshold set to 100MB")
print("Tables that will auto-broadcast:")
for name, df in [("products", df_products), ("customers", df_customers)]:
    # Estimate size (rough: row_count * avg_row_bytes)
    print(f"  {name}: ~{df.count() * len(df.columns) * 20 / (1024*1024):.1f} MB → BROADCAST")

In [0]:
print("BAD: Partitioning by customer_id")
bad_partition_count = df_fact.select("customer_id").distinct().count()
print(f"  Would create {bad_partition_count:,} partitions → tiny files, slow queries!")

In [0]:
from pyspark.sql.functions import date_format

df_partitioned = df_fact.withColumn("order_month", date_format("order_date", "yyyy-MM"))

good_partition_count = df_partitioned.select("order_month").distinct().count()
print(f"GOOD: Partitioning by order_month")
print(f"  Creates {good_partition_count} partitions → well-sized files, fast date filters!")

# Write with partitioning
target = f"{CATALOG}.gold.fact_sales_partitioned"
(df_partitioned.write
 .format("delta")
 .mode("overwrite")
 .partitionBy("order_month")
 .option("overwriteSchema", "true")
 .saveAsTable(target))

print(f"  Written to {target}")


In [0]:
print("Query: Filter for a specific month\n")

t1, _ = time_query("Non-partitioned (fact_sales)", lambda:
    spark.table(f"{CATALOG}.gold.fact_sales")
    .filter(date_format("order_date", "yyyy-MM") == "2024-06")
)

t2, _ = time_query("Partitioned (fact_sales_partitioned)", lambda:
    spark.table(f"{CATALOG}.gold.fact_sales_partitioned")
    .filter(col("order_month") == "2024-06")
)

print(f"\n  Partition pruning speedup: {t1/t2:.2f}x faster")


In [0]:
print("Current partition distribution of fact_sales:\n")

partition_stats = (df_fact
    .withColumn("partition_id", spark_partition_id())
    .groupBy("partition_id")
    .agg(count("*").alias("row_count"))
    .orderBy("partition_id")
)

total_partitions = partition_stats.count()
stats = partition_stats.agg(
    _sum("row_count").alias("total_rows"),
    avg("row_count").alias("avg_per_partition"),
    _sum("row_count").alias("total"),
).collect()[0]

print(f"  Total partitions: {total_partitions}")
print(f"  Avg rows/partition: {stats['avg_per_partition']:,.0f}")


In [0]:
print("COALESCE — reduce file count without shuffle\n")

before_parts = df_fact.select(spark_partition_id()).distinct().count()
print(f"  Before: {before_parts} partitions")

# Coalesce to fewer partitions (good for writing fewer, larger files)
df_coalesced = df_fact.coalesce(4)
after_parts = df_coalesced.select(spark_partition_id()).distinct().count()
print(f"  After coalesce(4): {after_parts} partitions")
print(f"  No shuffle occurred — just merged existing partitions")


In [0]:
print("REPARTITION — redistribute with full shuffle\n")

# Repartition by a column (useful before joins to co-locate data)
df_repartitioned = df_fact.repartition(8, "customer_state")

# Check distribution after repartition
repartitioned_stats = (df_repartitioned
    .withColumn("partition_id", spark_partition_id())
    .groupBy("partition_id")
    .agg(count("*").alias("row_count"))
    .orderBy("partition_id")
)

print(f"  Repartitioned to 8 partitions by customer_state:")
repartitioned_stats.show()
print("  Data is now co-located by state — joins/aggregations on state are faster")


In [0]:


df_agg_base = (df_fact
    .groupBy("customer_state", "product_category")
    .agg(
        _sum("line_total").alias("revenue"),
        count("order_item_id").alias("items"),
    )
)

print("WITHOUT CACHE — 3 queries on the same aggregation:\n")

t1, _ = time_query("Query 1 (top states)", lambda:
    df_agg_base.orderBy(col("revenue").desc()).limit(10)
)
t2, _ = time_query("Query 2 (top categories)", lambda:
    df_agg_base.groupBy("product_category").agg(_sum("revenue")).limit(10)
)
t3, _ = time_query("Query 3 (filter state)", lambda:
    df_agg_base.filter(col("customer_state") == "MH")
)

total_no_cache = t1 + t2 + t3
print(f"\n  Total without cache: {total_no_cache:.2f}s")

In [0]:
df_agg_cached = df_agg_base
# Materialize the cache (first action triggers caching)
df_agg_cached.count()

print("WITH CACHE — same 3 queries:\n")

t1, _ = time_query("Query 1 (top states)", lambda:
    df_agg_cached.orderBy(col("revenue").desc()).limit(10)
)
t2, _ = time_query("Query 2 (top categories)", lambda:
    df_agg_cached.groupBy("product_category").agg(_sum("revenue")).limit(10)
)
t3, _ = time_query("Query 3 (filter state)", lambda:
    df_agg_cached.filter(col("customer_state") == "MH")
)

total_cached = t1 + t2 + t3
print(f"\n  Total with cache: {total_cached:.2f}s")
print(f"  Speedup: {total_no_cache/total_cached:.2f}x faster")